# 46 — Learned numeric imputation

**Decision:** retain the original amount handling. Spatial height plus fold-fitted log-amount reconstruction lifts the standalone forest by 0.051 points, but every fixed ensemble variant is weaker than the promoted vote.

In [1]:
from pathlib import Path
import json
import pandas as pd
PROJECT_DIR = Path('../..').resolve()
OUTPUT_DIR = PROJECT_DIR / '.runtime' / 'learned-numeric-imputation-screen'
summary = pd.read_csv(OUTPUT_DIR / 'candidate-summary.csv', index_col='candidate')
result = json.loads((OUTPUT_DIR / 'result.json').read_text(encoding='utf-8'))
result

{'engineered_features': 29,
 'transformed_features_fold_1': 300,
 'learned_numeric_bag_accuracy': 0.8169191919191918,
 'change_vs_promoted_identity_vote': -0.0004840067340068366,
 'change_vs_spatial_height_bag': -0.0007575757575758457,
 'passes_gate': False,
 'local_test_opened': False,
 'competition_predictions_generated': False,
 'external_evidence': 'https://github.com/drivendataorg/pump-it-up/tree/master/mrbeer'}

## Course-aligned lifecycle

1. **Define the goal and scope** — test target-free numeric reconstruction without changing the classification objective.
2. **Gather the data** — use the frozen development predictors only.
3. **Explore the data** — treat non-positive amount as unavailable while retaining its recorded-state flag.
4. **Clean and preprocess the data** — reconstruct spatial height before log amount.
5. **Select and engineer features** — regress amount from the remaining compact numeric and frequency features.
6. **Define the machine-learning task** — retain three-class accuracy; regression is auxiliary preprocessing.
7. **Partition the data** — fit every reconstruction stage inside each outer-training partition.
8. **Select and train candidate methods** — refit accepted tree specifications and fixed votes.
9. **Evaluate and interpret the results** — compare component substitutions and one representation bag.
10. **Deploy and iterate** — reject promotion and stop amount-imputation tuning.

In [2]:
display_columns = ['mean_accuracy', 'leader_change', 'fold_wins_vs_leader', 'worst_fold_change', 'repair_recall_change', 'passes_gate']
summary.loc[:, display_columns].style.format({
    'mean_accuracy': '{:.4%}',
    'leader_change': '{:+.4%}',
    'worst_fold_change': '{:+.4%}',
    'repair_recall_change': '{:+.4%}',
})

,mean_accuracy,leader_change,fold_wins_vs_leader,worst_fold_change,repair_recall_change,passes_gate
candidate,,,,,,
spatial_height_representation_bag,81.7677%,+0.0274%,4,-0.0526%,+0.0869%,False
promoted_identity_vote,81.7403%,+0.0000%,0,+0.0000%,+0.0000%,False
learned_numeric_forest_vote,81.7003%,-0.0400%,1,-0.1052%,-0.4054%,False
learned_numeric_representation_bag,81.6919%,-0.0484%,1,-0.1473%,-0.2026%,False
learned_numeric_xgboost_vote,81.6667%,-0.0737%,1,-0.1263%,-0.2605%,False
learned_numeric_full_vote,81.6288%,-0.1115%,1,-0.2630%,-0.4053%,False
accepted_xgboost,81.0143%,-0.7260%,0,-0.8733%,-2.4034%,False
learned_numeric_xgboost,80.9996%,-0.7407%,0,-1.0311%,-1.5634%,False
learned_numeric_random_forest,80.6418%,-1.0985%,0,-1.3258%,+2.7210%,False


## Interpretation

The learned Random Forest alone improves by 0.0505 percentage points, but its errors do not complement the promoted vote. Replacing both components loses 0.1115 points and breaches the worst-fold guard. A zero amount appears to retain useful acquisition or operating context that a plausible reconstructed value partly obscures. No threshold, regressor family or blend weight is tuned, and the local test remains closed.